In [ ]:
!pip install tavily-python

In [27]:
import asyncio
from typing import Literal, Any
from tavily import AsyncTavilyClient


async def tavily_search(
    queries: list[str],
    api_key: str,
    max_results: int = 5,   
    search_depth: Literal["basic", "advanced"] = "basic",
    include_domains: list[str] | None = None,
    exclude_domains: list[str] | None = None,
    include_answer: bool = False,
    include_raw_content: bool|str = "markdown"
) -> list[dict[str, Any]]:
    """
    Execute Tavily search for multiple queries and return raw results.
    """

    client = AsyncTavilyClient(api_key=api_key)

    async def _search_one(query: str):
        return await client.search(
            query=query,
            max_results=max_results,
            search_depth=search_depth,
            include_domains=include_domains,
            exclude_domains=exclude_domains,
            include_answer=include_answer,
            include_raw_content="markdown",
        )

    tasks = [_search_one(q) for q in queries]
    return await asyncio.gather(*tasks)


In [33]:
def extract_raw_contents(
    tavily_responses: list[dict[str, Any]],
    max_chars_per_doc: int = 1000,
) -> list[str]:
    """
    Extract and truncate raw_content from Tavily responses.
    """
    documents = []

    for response in tavily_responses:
        for r in response.get("results", []):
            raw = r.get("raw_content")
            if not raw:
                continue

            raw = raw.strip()
            if len(raw) > max_chars_per_doc:
                raw = raw[:max_chars_per_doc]

            documents.append(raw)

    return documents


In [ ]:
!pip install --upgrade openai

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

def run_llm(prompt: str, model: str = "gpt-4.1-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful research assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.01,
    )
    return response.choices[0].message.content

In [ ]:
queries = [
    "scientific document understanding NLP",
    "discourse parsing academic papers",
    "information extraction scientific documents",
]

user_question = """
What NLP techniques allow deep reasoning over scientific papers,
including discourse structure and cross-section dependencies?
"""

# 1. Search
responses = await tavily_search(
    max_results=20,
    queries=queries,
    api_key="",
    search_depth="advanced",
    include_domains = ["https://aclanthology.org"],
)

# 2. Extract raw content
raw_docs = extract_raw_contents(responses)


In [71]:
print(responses)

[{'query': 'scientific document understanding NLP', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://aclanthology.org/2024.acl-demos.13.pdf', 'title': 'NLP-KG: A System for Exploratory Search of Scientific ...', 'content': 'investigating the relation-ships between different fields, understanding unfamiliar concepts in NLP, and finding rel-evant research literature. Demo, video, and code are available at: [...] Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 3: System Demonstrations), pages 127–135 August 11-16, 2024 ©2024 Association for Computational Linguistics NLP-KG: A System for Exploratory Search of Scientific Literature in Natural Language Processing Tim Schopf and Florian Matthes Technical University of Munich, Department of Computer Science, Germany {tim.schopf,matthes}@tum.de Abstract Scientific literature searches are often ex-ploratory, [...] whereby users are not yet familiar with a part

In [24]:
print(raw_docs)

['Proceedings of the Fifth Workshop on Scholarly Document Processing (SDP 2025), pages 1–6 July 31, 2025 ©2025 Association for Computational Linguistics Overview of the Fifth Workshop on Scholarly Document Processing Tirthankar Ghosala Philipp Mayrb Anita de Waardc Aakanksha Naikd Amanpreet Singhd Dayne Freitage Georg Rehmf,g Sonja Schimmlerh,i Dan Lic Abstract The workshop on Scholarly Document Pro-cessing (SDP) started in 2020 to accelerate re-search, inform policy, and educate the pub-lic on natural language processing for scien-tific text.\nThe fifth iteration of the work-shop, SDP 2025 was held at the 63rd An-nual Meeting of the Association for Compu-tational Linguistics (ACL 2025) in Vienna as a hybrid event.\nThe workshop saw a great increase in interest, with 26 submissions, of which 11 were accepted for the research track.\nThe program consisted of a research track, in-vited talks and four shared tasks: (1) SciHal25: Hallucination Detection for Scientific Content, (2) SciVQA: 

In [72]:
import re

def extract_evidence(text, meta_chars=400, abs_chars=600):
    """
    Lấy:
    - meta_chars ký tự trước 'Abstract'
    - abs_chars ký tự sau 'Abstract'
    """
    m = re.search(r'\babstract\b', text, re.I)

    if not m:
        return text[:meta_chars + abs_chars]

    meta = text[max(0, m.start() - meta_chars):m.start()]
    abstract = text[m.end():m.end() + abs_chars]

    return (
        "Metadata:\n" + meta.strip() +
        "\n\nAbstract:\n" + abstract.strip()
    )


In [66]:
def select_relevant_docs(raw_docs, question, k=10):
    snippets = [extract_evidence(d) for d in raw_docs]

    prompt = f"""
You are selecting relevant scientific documents.

Research question:
{question}

Select the {k} most relevant documents.

IMPORTANT:
- Return ONLY a JSON array of integers
- No explanation
- Indices must refer to the document numbers below

Documents:
""" + "\n\n".join(
        f"[{i}] {s}" for i, s in enumerate(snippets)
    )

    resp = run_llm(prompt, model="gpt-4.1-mini")

    try:
        indices = eval(resp.strip())
        assert isinstance(indices, list)
    except:
        raise ValueError(f"Invalid LLM output: {resp}")

    return [raw_docs[i] for i in indices[:k]]


In [73]:
relevant_docs = select_relevant_docs(raw_docs, user_question)
print(relevant_docs)

['Proceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 3: System Demonstrations), pages 127–135 August 11-16, 2024 ©2024 Association for Computational Linguistics NLP-KG: A System for Exploratory Search of Scientific Literature in Natural Language Processing Tim Schopf and Florian Matthes Technical University of Munich, Department of Computer Science, Germany {tim.schopf,matthes}@tum.de Abstract Scientific literature searches are often ex-ploratory, whereby users are not yet familiar with a particular field or concept but are in-terested in learning more about it. However, existing systems for scientific literature search are typically tailored to keyword-based lookup searches, limiting the possibilities for explo-ration. We propose NLP-KG, a feature-rich sys-tem designed to support the exploration of re-search literature in unfamiliar natural language processing (NLP) fields. In addition to a se-mantic search, NLP-KG allows users to easily fi

In [74]:
def build_prompt(relevant_docs, question):
    docs_block = "\n\n".join(
        f"""[Document {i}]
{doc}
""" for i, doc in enumerate(relevant_docs)
    )

    prompt = f"""
You are a research assistant.

Task:
Answer the following research question using ONLY the information
from the provided documents.

If the documents do not contain sufficient information,
explicitly say: "The provided documents are insufficient to answer this question."

Research question:
{question}

Documents:
{docs_block}

Answer:
"""
    return prompt.strip()


In [75]:
# 3. Build prompt
prompt = build_prompt(relevant_docs, user_question)

# 4. Run LLM
answer = run_llm(prompt)

print(answer)

The provided documents describe several NLP techniques that enable deep reasoning over scientific papers, including discourse structure and cross-section dependencies:

1. **Discourse Parsing with Statistical and Recursive Deep Models**  
   - Document 3 presents a decision tree-based statistical method for discourse parsing that models semantic dependencies among sentences, capturing sentential dependencies probabilistically.  
   - Document 4 proposes recursive deep models for discourse parsing that jointly learn distributed representations for clauses, sentences, and entire discourses. This approach captures intentional, semantic, and syntactic aspects governing discourse coherence, enabling deeper understanding of discourse structure.

2. **Incorporation of Discourse and Lexical Constraints via Generalized Expectation (GE) Criterion**  
   - Document 2 introduces a method to improve information structure analysis of scientific documents by guiding feature-based machine learning mod